In [1]:
import json
import pandas as pd
from urlextract import URLExtract
from urllib.parse import urlparse
import subprocess
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from collections import Counter
import matplotlib.pyplot as plt

os.makedirs("graphs", exist_ok=True)

In [2]:
dic_human = {}
df_human = pd.read_csv("../data/CF/cf_v1_test.csv")
for _, row in df_human.iterrows():
    dic_human[row["claimId"]] = row["evidenceURLs"]

dic_gpt = json.load(open("../experiments/logs/web_search_gpt-5-nano_True_True.json", "r"))
dic_grok = json.load(open("../experiments/logs/web_search_grok-4.3_True_True.json", "r"))
dic_gemini = json.load(open("../experiments/logs/web_search_gemini-2.5-flash_True_True.json", "r"))

dic_gpt_gs = json.load(open("../experiments/logs/web_search_URL_gpt-5-nano_True_True.json", "r"))
dic_grok_gs = json.load(open("../experiments/logs/web_search_URL_grok-4.3_True_True.json", "r"))
dic_gemini_gs = json.load(open("../experiments/logs/web_search_URL_gemini-2.5-flash_True_True.json", "r"))

In [3]:
class humanExtractor:
    extractor = URLExtract()

    def extract_urls_resp(res_text):
        urls = humanExtractor.extractor.find_urls(res_text)
        urls = list(set(urls))
        urls = [urlparse(u).netloc for u in urls]

        return urls

    def extract_urls(dic):
        def process_one(item):
            i, data = item
            urls = humanExtractor.extract_urls_resp(data)
            return i, urls

        dic_urls = {}

        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = [
                executor.submit(process_one, item)
                for item in dic.items()
            ]

            for future in tqdm(as_completed(futures), total=len(futures)):
                i, urls = future.result()
                dic_urls[i] = urls

        return dic_urls
    
class modelExtractor:
    extractor = URLExtract()
    def get_redirect_url(url):
        try:
            response = subprocess.run(
                ["curl", "-Ls", "-o", "/dev/null", "-w", "%{url_effective}", url],
                capture_output=True,
                text=True,
                timeout=150
            )
        except:
            return None
        return response.stdout

    def extract_urls_resp(res_text, type):
        urls = modelExtractor.extractor.find_urls(res_text)

        if type == "gpt":
            urls = [url for url in urls if url.lower().startswith(("http://", "https://"))]

        if type == "gemini":
            urls = [modelExtractor.get_redirect_url(url) for url in urls if "https://vertexaisearch.cloud.google.com/grounding-api-redirect/" in url]

        urls = list(set(urls))
        urls = [urlparse(u).netloc for u in urls]

        return urls

    def extract_urls(dic, type):
        def process_one(item):
            i, data = item
            urls = modelExtractor.extract_urls_resp(data["response"], type)
            return i, urls


        dic_urls = {}

        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = [
                executor.submit(process_one, item)
                for item in dic.items()
            ]

            for future in tqdm(as_completed(futures), total=len(futures)):
                i, urls = future.result()
                dic_urls[i] = urls

        return dic_urls

In [4]:
human_urls = humanExtractor.extract_urls(dic_human)

gpt_urls = modelExtractor.extract_urls(dic_gpt, "gpt")
grok_urls = modelExtractor.extract_urls(dic_grok, "grok")
gem_urls = modelExtractor.extract_urls(dic_gemini, "gemini")

gpt_gs_urls = modelExtractor.extract_urls(dic_gpt_gs, "gpt")
grok_gs_urls = modelExtractor.extract_urls(dic_grok_gs, "grok")
gem_gs_urls = modelExtractor.extract_urls(dic_gemini_gs, "gemini")

100%|██████████| 3578/3578 [13:30<00:00,  4.41it/s]


In [5]:
def clean_domain(x):
    if not isinstance(x, str) or not x.strip():
        return None
    x = x.strip().lower()
    if x.startswith(("http://", "https://")):
        x = urlparse(x).netloc.lower()
    x = x.split("@")[-1].split(":")[0]
    if x.startswith("www."):
        x = x[4:]
    return x or None

human_domains = {i: [clean_domain(url) for url in urls] for i, urls in human_urls.items()}
gpt_domains = {i: [clean_domain(url) for url in urls] for i, urls in gpt_urls.items()}
grok_domains = {i: [clean_domain(url) for url in urls] for i, urls in grok_urls.items()}
gem_domains = {i: [clean_domain(url) for url in urls] for i, urls in gem_urls.items()}  
gpt_gs_domains = {i: [clean_domain(url) for url in urls] for i, urls in gpt_gs_urls.items()}
grok_gs_domains = {i: [clean_domain(url) for url in urls] for i, urls in grok_gs_urls.items()}
gem_gs_domains = {i: [clean_domain(url) for url in urls] for i, urls in gem_gs_urls.items()}

In [9]:
def hit_miss_counter(human_domains, model_domains):
    hit = 0
    miss = 0
    for i in human_domains:
        human_set = set(human_domains[i])
        model_set = set(model_domains[i])
        if human_set & model_set:
            hit += 1
        else:
            miss += 1
    return round(hit / (hit + miss), 2) if (hit + miss) > 0 else 0

print("GPT:", hit_miss_counter(human_domains, gpt_domains))
print("GPT + Guided Search:", hit_miss_counter(human_domains, gpt_gs_domains))

print("Grok:", hit_miss_counter(human_domains, grok_domains))
print("Grok + Guided Search:", hit_miss_counter(human_domains, grok_gs_domains))

print("Gemini:", hit_miss_counter(human_domains, gem_domains))
print("Gemini + Guided Search:", hit_miss_counter(human_domains, gem_gs_domains))

GPT: 0.13
GPT + Guided Search: 0.56
Grok: 0.62
Grok + Guided Search: 0.94
Gemini: 0.04
Gemini + Guided Search: 0.02
